好的，以下是《动手学深度学习》第11.8节 **RMSProp算法** 的学习笔记，已整理成 Markdown 格式，并补充代码运行示例，便于你学习和记录。

---

# 📘 11.8 RMSProp算法 - 学习笔记

## 📌 一、RMSProp 是什么？

**RMSProp（Root Mean Square Propagation）** 是对 AdaGrad 的改进版本，它通过\*\*滑动平均（指数加权平均）\*\*的方式，缓解了 AdaGrad **学习率快速衰减过小** 的问题。

> **本质：仍然是按维度自适应学习率的优化方法，只不过不再累加全部历史梯度，而是滑动平均近几步的梯度平方。**

---

## ⚙️ 二、算法原理（公式与解释）

### 🔢 更新公式：

$$
\boldsymbol{s}_t = \gamma \boldsymbol{s}_{t-1} + (1 - \gamma) \boldsymbol{g}_t^2
$$

$$
\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \frac{\eta}{\sqrt{\boldsymbol{s}_t + \epsilon}} \odot \boldsymbol{g}_t
$$

其中：

* \$\boldsymbol{s}\_t\$：历史梯度平方的**滑动平均**
* \$\gamma\$：衰减因子（一般为0.9）
* \$\eta\$：初始学习率
* \$\epsilon\$：小常数，防止除以0

### 🧠 直观理解：

* 不像 AdaGrad 那样一直累加历史梯度平方 → RMSProp 只看最近的趋势
* **频繁更新 → 梯度方差大 → 学习率自动缩小**
* **方向稳定 → 梯度方差小 → 学习率保持适中**

---

## 🧪 三、代码实现

### ✅ 从零实现（手动实现）：

```python
def rmsprop(params, states, hyperparams):
    gamma, eps = hyperparams['gamma'], 1e-6
    for p, s in zip(params, states):
        with torch.no_grad():
            s[:] = gamma * s + (1 - gamma) * torch.square(p.grad)
            p[:] -= hyperparams['lr'] * p.grad / torch.sqrt(s + eps)
        p.grad.zero_()
```

### 🔧 初始化状态：

```python
def init_rmsprop_states(feature_dim):
    return (torch.zeros((feature_dim, 1)), torch.zeros((1,)))
```

### 📦 简洁实现（PyTorch）：

```python
trainer = torch.optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9)
```

> 🔍 注意：`alpha` 在 PyTorch 中表示的是衰减因子 \$\gamma\$。

---

## 🧪 四、实验示例（使用二次函数）

书中例子通常会用一个简化的凸函数（如椭圆形等高线）验证优化器收敛效果：

```python
import torch
from d2l import torch as d2l

def f(x1, x2):
    return 0.1 * x1**2 + 2 * x2**2

def grad_f(x1, x2):
    return torch.tensor([0.2 * x1, 4 * x2])

d2l.set_figsize()
d2l.plot_traj(20, f, grad_f, init=(2, 10), optimizer=rmsprop,
              states=[torch.zeros(2)], hyperparams={'lr': 0.1, 'gamma': 0.9})
```

📈 输出会显示 RMSProp 如何调整方向、快速收敛。

---

## 📊 五、与 AdaGrad 对比

| 对比项    | AdaGrad    | RMSProp    |
| ------ | ---------- | ---------- |
| 历史梯度处理 | 全部累加（不可忘记） | 滑动平均（指数加权） |
| 学习率变化  | 越来越小       | 稳定在一个合适范围  |
| 易陷入停滞  | 是，后期几乎不更新  | 否，仍可保持适度更新 |
| 适用场景   | 稀疏特征，短期训练  | 更通用，适合长期训练 |

---

## 📝 六、总结要点

* RMSProp 是 AdaGrad 的改进版：引入了 **梯度平方的滑动平均**。
* 可以有效避免 **学习率衰减太快导致停止学习** 的问题。
* 在许多实际任务中都比 AdaGrad 更稳定，是 **实际训练中的主流选择之一**。

---

是否需要我继续整理第11.9节 **Adam优化算法** 的 Markdown 笔记？这是RMSProp与动量法的结合版，也是最常用的优化器之一。
